# ABM in 5 minutes

ABM is a deterministic memory runtime for compiled knowledge. Unlike vector memories, it predicts **in advance** how accurately it can answer under a fixed memory budget.

```bash
pip install abm-runtime
```

In [ ]:
from abm import Memory, contract, report

mem = Memory(dim=8192)

mem.store("payment_service", "requires", "auth_service")
mem.store("auth_service", "writes_to", "session_store")
mem.store("payment_service", "owned_by", "team_payments")
mem.store("team_payments", "on_call", "alice")

## Ask — one hop, many hops

In [ ]:
print(mem.query("payment_service", "requires"))
print(mem.chain("payment_service", ["owned_by", "on_call"]))
print(mem.member("auth_service", "writes_to", "session_store"))

Confidence is **calibrated**: 0.5 means "indistinguishable from random". Inverse questions come free — a stored fact answers in both directions.

## The Memory Contract — computed, not measured

In [ ]:
print(contract(mem, grounding=0.93))

Every number above comes from a formula, not a benchmark. `grounding` is the audited precision of whatever produced your facts (an LLM extractor, a parser) — the contract projects the end-to-end accuracy.

## Verify the promise

Load the memory with more facts, then check the contract against reality:

In [ ]:
facts = [(f"svc_{i}", f"rel_{i % 11}", f"obj_{i}") for i in range(150)]
big = Memory(dim=4096)
for f in facts:
    big.store(*f)

promised = big.expected_accuracy()
measured = sum(big.query(s, r)[0] == o for s, r, o in facts) / len(facts)
print(f"promised {promised:.1%}   measured {measured:.1%}")

If those two numbers disagree by more than a few points, file a bug — the contract is the product.

## Diagnostics

`report()` shows the full panel: capacity, pressure, recommended size, dominant bottleneck, max reasoning depth. From the shell: `abm inspect your_triples.json --grounding 0.9`.

In [ ]:
print(report(big))